# Segment 5 — 2.5D Semantic Elevation Map

This notebook implements the fixed-resolution map described in the supplied Segment 5 notes:

**Classified LiDAR points → 2.5D grid**

Each cell stores:
- elevation
- minimum/maximum height
- semantic class
- semantic confidence
- occupancy
- dynamic probability
- point count
- timestamp

The adaptive/foveated resolution is intentionally **not** implemented here; that belongs to Segment 6.


In [ ]:
# 1. Imports and configuration

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from collections import defaultdict
from datetime import datetime

# Fixed resolution for Segment 5
RESOLUTION = 0.05       # 5 cm
MAP_SIZE_X = 100.0      # metres
MAP_SIZE_Y = 100.0      # metres

# Coordinate convention:
# x = forward
# y = left/right
# z = height

print(f"Resolution: {RESOLUTION} m")
print(f"Map size: {MAP_SIZE_X} m x {MAP_SIZE_Y} m")


In [ ]:
# 2. MapCell data structure

@dataclass
class MapCell:
    elevation: float = np.nan
    min_height: float = np.nan
    max_height: float = np.nan

    semantic_class: str = "UNKNOWN"
    semantic_confidence: float = 0.0

    occupancy: float = 0.0
    dynamic_probability: float = 0.0

    point_count: int = 0
    timestamp: float = 0.0

    # Internal accumulators
    heights: list = field(default_factory=list)
    semantic_votes: dict = field(default_factory=lambda: defaultdict(float))
    dynamic_votes: list = field(default_factory=list)

    def add_point(self, z, label="UNKNOWN", confidence=1.0,
                  dynamic_probability=0.0, timestamp=None):
        self.heights.append(float(z))

        # Confidence-weighted semantic voting
        self.semantic_votes[str(label)] += float(confidence)

        self.dynamic_votes.append(float(dynamic_probability))

        self.point_count += 1

        if timestamp is not None:
            self.timestamp = float(timestamp)

    def finalize(self):
        if not self.heights:
            return

        z = np.asarray(self.heights, dtype=float)

        self.min_height = float(np.min(z))
        self.max_height = float(np.max(z))

        # Mean elevation for the first implementation
        self.elevation = float(np.mean(z))

        # Confidence-weighted semantic class
        if self.semantic_votes:
            self.semantic_class = max(
                self.semantic_votes,
                key=self.semantic_votes.get
            )

            total = sum(self.semantic_votes.values())
            self.semantic_confidence = (
                self.semantic_votes[self.semantic_class] / total
                if total > 0 else 0.0
            )

        # Average dynamic probability
        if self.dynamic_votes:
            self.dynamic_probability = float(np.mean(self.dynamic_votes))

        # Simple occupancy probability
        self.occupancy = 1.0


In [ ]:
# 3. Grid creation

class SemanticElevationGrid:
    def __init__(self, size_x, size_y, resolution):
        self.size_x = float(size_x)
        self.size_y = float(size_y)
        self.resolution = float(resolution)

        self.width = int(np.ceil(size_x / resolution))
        self.height = int(np.ceil(size_y / resolution))

        # Dictionary is memory-efficient for sparse LiDAR observations.
        self.cells = {}

    def world_to_grid(self, x, y):
        gx = int(np.floor(x / self.resolution))
        gy = int(np.floor(y / self.resolution))

        return gx, gy

    def grid_to_world(self, gx, gy):
        x = (gx + 0.5) * self.resolution
        y = (gy + 0.5) * self.resolution

        return x, y

    def get_cell(self, gx, gy):
        if not (0 <= gx < self.width and 0 <= gy < self.height):
            return None

        key = (gx, gy)

        if key not in self.cells:
            self.cells[key] = MapCell()

        return self.cells[key]

    def add_point(self, point, label="UNKNOWN", confidence=1.0,
                  dynamic_probability=0.0, timestamp=None):
        x, y, z = point[:3]

        gx, gy = self.world_to_grid(x, y)
        cell = self.get_cell(gx, gy)

        if cell is not None:
            cell.add_point(
                z=z,
                label=label,
                confidence=confidence,
                dynamic_probability=dynamic_probability,
                timestamp=timestamp
            )

    def finalize(self):
        for cell in self.cells.values():
            cell.finalize()

    def observed_mask(self):
        mask = np.zeros((self.height, self.width), dtype=bool)

        for gx, gy in self.cells:
            mask[gy, gx] = True

        return mask

print("Grid class ready.")


In [ ]:
# 4. Example classified LiDAR data

# In the real project, replace this with the output from Segment 4.
#
# Columns:
# x, y, z, label, confidence, dynamic_probability

rng = np.random.default_rng(42)

points = []
labels = []
confidences = []
dynamic_probs = []

# Road / ground
for _ in range(3000):
    x = rng.uniform(0, 40)
    y = rng.uniform(-15, 15)
    z = 0.02 * rng.normal()

    points.append([x, y, z])
    labels.append("ROAD")
    confidences.append(rng.uniform(0.90, 0.99))
    dynamic_probs.append(0.01)

# Vehicle
for _ in range(600):
    x = rng.uniform(12, 17)
    y = rng.uniform(-3, 1)
    z = rng.uniform(0.05, 1.7)

    points.append([x, y, z])
    labels.append("VEHICLE")
    confidences.append(rng.uniform(0.85, 0.98))
    dynamic_probs.append(0.90)

# Pedestrian
for _ in range(250):
    x = rng.uniform(22, 23.5)
    y = rng.uniform(5, 6)
    z = rng.uniform(0.05, 1.8)

    points.append([x, y, z])
    labels.append("PEDESTRIAN")
    confidences.append(rng.uniform(0.80, 0.97))
    dynamic_probs.append(0.85)

# Pole
for _ in range(200):
    x = rng.uniform(30, 30.5)
    y = rng.uniform(-7, -6.5)
    z = rng.uniform(0.05, 5.0)

    points.append([x, y, z])
    labels.append("POLE")
    confidences.append(rng.uniform(0.80, 0.98))
    dynamic_probs.append(0.01)

points = np.asarray(points, dtype=float)
confidences = np.asarray(confidences)
dynamic_probs = np.asarray(dynamic_probs)

print("Number of points:", len(points))
print("Point array shape:", points.shape)


In [ ]:
# 5. Build the 2.5D map

grid = SemanticElevationGrid(
    size_x=MAP_SIZE_X,
    size_y=MAP_SIZE_Y,
    resolution=RESOLUTION
)

timestamp = datetime.now().timestamp()

for point, label, confidence, dynamic_probability in zip(
    points, labels, confidences, dynamic_probs
):
    grid.add_point(
        point=point,
        label=label,
        confidence=confidence,
        dynamic_probability=dynamic_probability,
        timestamp=timestamp
    )

grid.finalize()

print("Observed cells:", len(grid.cells))
print("Total possible cells:", grid.width * grid.height)


In [ ]:
# 6. Extract numerical map arrays

elevation = np.full((grid.height, grid.width), np.nan)
min_height = np.full((grid.height, grid.width), np.nan)
max_height = np.full((grid.height, grid.width), np.nan)
occupancy = np.zeros((grid.height, grid.width))
dynamic_probability = np.zeros((grid.height, grid.width))
confidence_map = np.zeros((grid.height, grid.width))
point_count = np.zeros((grid.height, grid.width), dtype=int)

semantic_map = np.full(
    (grid.height, grid.width),
    "UNKNOWN",
    dtype=object
)

for (gx, gy), cell in grid.cells.items():
    elevation[gy, gx] = cell.elevation
    min_height[gy, gx] = cell.min_height
    max_height[gy, gx] = cell.max_height

    occupancy[gy, gx] = cell.occupancy
    dynamic_probability[gy, gx] = cell.dynamic_probability
    confidence_map[gy, gx] = cell.semantic_confidence
    point_count[gy, gx] = cell.point_count

    semantic_map[gy, gx] = cell.semantic_class

print("Elevation map shape:", elevation.shape)
print("Observed cells:", np.sum(~np.isnan(elevation)))


In [ ]:
# 7. Height range / geometric obstacle information

height_range = max_height - min_height

# Simple geometric flag.
# This is NOT a replacement for AI segmentation.
OBSTACLE_HEIGHT_THRESHOLD = 0.30

possible_obstacle = (
    np.isfinite(height_range) &
    (height_range > OBSTACLE_HEIGHT_THRESHOLD)
)

print("Possible obstacle cells:", np.sum(possible_obstacle))


In [ ]:
# 8. Visualize elevation

plt.figure(figsize=(10, 7))

masked_elevation = np.ma.masked_invalid(elevation)

plt.imshow(
    masked_elevation,
    origin="lower",
    interpolation="nearest"
)

plt.colorbar(label="Elevation (m)")
plt.title("2.5D Elevation Map")
plt.xlabel("Grid X")
plt.ylabel("Grid Y")
plt.show()


In [ ]:
# 9. Visualize semantic classes

class_to_id = {
    "UNKNOWN": 0,
    "ROAD": 1,
    "VEHICLE": 2,
    "PEDESTRIAN": 3,
    "POLE": 4
}

semantic_ids = np.zeros_like(elevation, dtype=float)

for class_name, class_id in class_to_id.items():
    semantic_ids[semantic_map == class_name] = class_id

semantic_ids[np.isnan(elevation)] = np.nan

plt.figure(figsize=(10, 7))

plt.imshow(
    np.ma.masked_invalid(semantic_ids),
    origin="lower",
    interpolation="nearest"
)

plt.colorbar(
    ticks=list(class_to_id.values()),
    label="Semantic class ID"
)

plt.title("2.5D Semantic Map")
plt.xlabel("Grid X")
plt.ylabel("Grid Y")
plt.show()


In [ ]:
# 10. Visualize occupancy and dynamic probability

plt.figure(figsize=(10, 7))

plt.imshow(
    np.ma.masked_where(elevation != elevation, occupancy),
    origin="lower",
    interpolation="nearest",
    vmin=0,
    vmax=1
)

plt.colorbar(label="Occupancy probability")
plt.title("Occupancy Map")
plt.xlabel("Grid X")
plt.ylabel("Grid Y")
plt.show()


In [ ]:
# 11. Dynamic probability map

plt.figure(figsize=(10, 7))

plt.imshow(
    np.ma.masked_where(elevation != elevation, dynamic_probability),
    origin="lower",
    interpolation="nearest",
    vmin=0,
    vmax=1
)

plt.colorbar(label="Dynamic probability")
plt.title("Dynamic Probability Map")
plt.xlabel("Grid X")
plt.ylabel("Grid Y")
plt.show()


In [ ]:
# 12. Query a specific world coordinate

def inspect_world_position(x, y):
    gx, gy = grid.world_to_grid(x, y)

    cell = grid.get_cell(gx, gy)

    if cell is None or cell.point_count == 0:
        return {
            "cell": [gx, gy],
            "observed": False,
            "semantic": "UNKNOWN"
        }

    return {
        "cell": [gx, gy],
        "observed": True,
        "elevation": cell.elevation,
        "min_height": cell.min_height,
        "max_height": cell.max_height,
        "semantic": cell.semantic_class,
        "confidence": cell.semantic_confidence,
        "occupancy": cell.occupancy,
        "dynamic_probability": cell.dynamic_probability,
        "points": cell.point_count,
        "timestamp": cell.timestamp
    }

print(inspect_world_position(14.0, -1.0))


In [ ]:
# 13. Export cells to JSON-compatible records

def grid_to_records(grid):
    records = []

    for (gx, gy), cell in grid.cells.items():
        records.append({
            "cell": [gx, gy],
            "elevation": cell.elevation,
            "min_height": cell.min_height,
            "max_height": cell.max_height,
            "semantic": cell.semantic_class,
            "confidence": cell.semantic_confidence,
            "occupancy": cell.occupancy,
            "dynamic_probability": cell.dynamic_probability,
            "points": cell.point_count,
            "timestamp": cell.timestamp
        })

    return records

records = grid_to_records(grid)

print("Number of exported cells:", len(records))
print(records[0])


## 14. Replace the demo data with Segment 4 output

Your real pipeline should eventually look like:

```text
RAW LiDAR
   ↓
Segment 2 — Cleaning
   ↓
Segment 3 — Point classification
   ↓
Segment 4 — Objects + terrain
   ↓
THIS NOTEBOOK — Segment 5
   ↓
Fixed-resolution 2.5D semantic elevation map
   ↓
Segment 6 — Adaptive / foveated resolution
```

For the real system, replace the synthetic `points`, `labels`, `confidences`, and `dynamic_probs` with the outputs of Segment 4.

Important: this notebook intentionally uses a **uniform 5 cm grid**. The variable resolutions such as 5 cm / 10 cm / 25 cm / 50 cm should be implemented in Segment 6.
